# Easy Reads Project Workflow

This notebook documents the project workflow implemented in the current codebase. It follows the same order as the runtime pipeline in `main_easy_reads.py`, then expands each module with the functions used at that stage and a short description of what each function does.

## End-to-End Workflow

1. `main_easy_reads.main()` starts the pipeline and reads the user configuration.
2. `paper_downloader.download_file()` downloads the ArXiv source archive.
3. `paper_downloader.extract_tar()` extracts the source files into the working folder.
4. `paper_downloader.find_largest_tex()` selects the most likely main LaTeX file.
5. `paper_downloader.replace_missing_document_class()` swaps a journal-specific class for `article` when the class file is unavailable.
6. `paper_downloader.ensure_tuning_block()` inserts the auto-tuning LaTeX block into the chosen TeX file.
7. `paper_tuner.set_tuning_values_newfile()` updates the tuning block with the configured font, spacing, figure-width, and caption values.
8. `paper_tex_to_pdf.compile_with_multiple_passes()` runs the compilation workflow and retries if a missing class blocks compilation.
9. Inside compilation, `paper_tex_to_pdf.compile_latex_to_pdf()` performs the actual pdflatex and bibtex passes.
10. During failed compilation analysis, `paper_tex_to_pdf.check_and_suggest_missing_class()` inspects the LaTeX log and reports missing class files with suggested fixes.

## Module 1: `main_easy_reads.py`

### Function order in workflow

1. `main()`
   Orchestrates the full pipeline. It downloads the paper source, extracts it, finds the main TeX file, applies class and tuning adjustments, then compiles the final PDF. It also uses the user-configured values `url`, `folder`, `base_font_pt`, `baseline_pt`, `default_fig_width`, and `caption_size`.

## Module 2: `paper_downloader.py`

### Function order in workflow

1. `download_file(url, save_as=None)`
   Downloads the remote file with `requests`, infers a filename from the response headers or URL when needed, and writes the archive to disk.

2. `extract_tar(filename, extract_to='.')`
   Opens the downloaded archive with `tarfile` and extracts all contents into the target directory.

3. `find_largest_tex(root_folder)`
   Walks the extracted directory tree, finds `.tex` files, prefers original files over `_easy.tex` versions, and returns the largest candidate as the likely main paper source.

4. `replace_missing_document_class(tex_path, force_replace=False)`
   Reads the selected TeX file, detects journal-specific document classes, checks whether the corresponding `.cls` file exists locally or system-wide, and if not, writes an `_easy.tex` version that replaces the class with `article`. This is called once before compilation and may also be called again later as a forced retry after a LaTeX failure.

5. `ensure_tuning_block(tex_path)`
   Inserts the predefined LaTeX tuning block after `\documentclass` if it is not already present, then writes or updates the `_easy.tex` file that downstream steps modify.

### Supporting constant

- `TUNING_BLOCK`
  The LaTeX snippet injected into the paper. It defines font-size, baseline-skip, figure-width, and caption-size commands and applies them at document start.

## Module 3: `paper_tuner.py`

### Function order in workflow

1. `set_tuning_values_newfile(tex_path, base_font_pt=None, baseline_pt=None, default_fig_width=None, caption_size=None)`
   Locates the inserted tuning block inside the TeX file and rewrites its parameter values to match the user configuration. It then saves the updated result back to the `_easy.tex` file that will be compiled.

## Module 4: `paper_tex_to_pdf.py`

### Function order in workflow

1. `compile_with_multiple_passes(tex_file_path, output_dir=None)`
   Controls the overall LaTeX compilation strategy. It runs an initial compilation, checks for failure, optionally forces a document-class replacement if a missing `.cls` file is detected, and then reruns compilation to improve citation resolution.

2. `compile_latex_to_pdf(tex_file_path, output_dir=None, max_attempts=4)`
   Performs the actual command-line compilation work. It changes into the TeX directory, runs `pdflatex`, conditionally runs `bibtex` when a bibliography is found, runs additional `pdflatex` passes, checks log and aux files for errors or unresolved references, and returns the generated PDF path when successful.

3. `check_and_suggest_missing_class(log_content, output_dir=None)`
   Parses LaTeX log content for missing `.cls` files, checks whether the class exists in the extracted directory, and prints practical suggestions for recovery. This function is used during compilation failure handling rather than at the main-entry stage.

### Supporting constant

- `JOURNAL_CLASS_MAPPING`
  Maps common journal class files to the fallback `article` class so error messages and remediation steps are easier to explain.

## Practical Notes

- The pipeline assumes the ArXiv source archive is available from the `src` endpoint, not the abstract page.
- The project depends on a working LaTeX installation that provides `pdflatex` and usually `bibtex`.
- The `_easy.tex` file is the project's edited working copy; the original paper source is preserved when possible.
- `compile_with_multiple_passes()` may invoke `replace_missing_document_class()` a second time when the first compilation proves that a journal class is missing from the local LaTeX installation.